# ISEL - LEIM - PIV
## 5.º Semestre Letivo 2025/2026
# <br>
###  <center> 2º Trabalho Prático </center>
# </br>
Trabalho realizado por:
* Diogo Santiago, número 51998
<b> Turma 52D - Docente: João Pedro Costa</b>

<b> Data de entrega: 07/12/2025 </b> 

In [ ]:
import cv2
import numpy as np
import math

#FUNÇÕES AUXILIARES
# Função para dividir um segmento de reta em partes iguais
# É usada para calcular os pontos que dividem a estrada em 3 faixas (esquerda, meio, direita)
def get_split(start, end, parts, idx):
    return int(start + (end - start) * (idx / parts))

# Função para gerir a "memória" do sistema de tracking
# Percorre todos os objetos atualmente em seguimento e remove aqueles que:
# 1 Falharam a deteção consecutiva num número excessivo de frames (max_missing).
# 2 Já saíram fisicamente da área visível da imagem (ultrapassaram o limite inferior).
def cleanup_objects(objects, max_missing, img_height):
    new_objects = {}
    for k, v in objects.items():
        # Se o objeto não foi detetado há demasiado tempo, é considerado ruiido
        if v['missing'] > max_missing: continue
        # Se o centro do objeto já passou o fundo da imagem, saiu da cena
        if v['center'][1] > (img_height - 10): continue
        # Caso contrário, mantém o objeto ativo
        new_objects[k] = v
    return new_objects

#FUNÇÃO PRINCIPAL DE TRACKING
# Implementa a logica de associação de dados entre frames consecutivos
def update_tracker(objects, detections, frame_count, fps, pixels_per_meter, img_height, next_id, total_count):
    max_dist = 50      # limiar de distância máxima para associação em pxs
    max_missing = 3    # frames tolerados sem deteção
    
    # Se não existem objetos em memória, todas as novas deteções são novos veículos
    if not objects:
        for det in detections:
            objects[next_id] = {
                'center': (det[0], det[1]),  # Centro (x, y)
                'rect': det[2:],             # Bounding Box (x, y, w, h)
                'start_pos': (det[0], det[1]), # Posição inicial para cálculo de velocidade
                'start_frame': frame_count,    # Frame de entrada
                'speed': 0.0,                # Velocidade 
                'missing': 0,                # Contador de falhas de deteção
                'confirmed': False,          # Flag de contagem (passou a linha virtual?)
                'processed_for_avg': False   # Flag para estatística de média
            }
            next_id += 1
        return objects, next_id, total_count

    object_ids = list(objects.keys())
    
   
    # Se não houver deteções neste frame, incrementa-se o contador de "missing" para todos
    if len(detections) == 0:
        for oid in object_ids:
            objects[oid]['missing'] += 1
        objects = cleanup_objects(objects, max_missing, img_height)
        return objects, next_id, total_count

    # MATRIZ DE DISTÂNCIAS
    # Matriz onde calcula a distância entre todos os objetos antigos e todas as novas deteções. Se do antigo para o novo mudou 50px é o mesmo carro 
    # Calcula a distância entre todos os objetos antigos e todas as novas deteções.
    # para todos com todos
    distances = np.zeros((len(object_ids), len(detections)))
    for i, oid in enumerate(object_ids):
        old_cx, old_cy = objects[oid]['center']
        for j, (new_cx, new_cy, _, _, _, _) in enumerate(detections):
            distances[i, j] = math.hypot(old_cx - new_cx, old_cy - new_cy) # distancia eucladiana

    used_rows = set() # Objetos antigos ja associados
    used_cols = set() # Novas deteções ja associadas

    # ALGORITMODE ASSOCIAÇÃO
    # Seleciona iterativamente o par (Objeto Antigo, Nova Deteção) com a menor distância
    if distances.size > 0:
        while len(used_rows) < len(object_ids) and len(used_cols) < len(detections):
            # Encontra o mínimo global na matriz de distâncias
            min_idx = np.unravel_index(np.argmin(distances), distances.shape)
            r, c = min_idx 
            
            # Se a distância mínima exceder o limiar, não é o mesmo objeto
            if distances[r, c] > max_dist: break
            
            if distances[r, c] != float('inf'):
                obj_id = object_ids[r]
                det = detections[c]
                
                # ATUALIZAÇÃO DO ESTADO DO OBJETO
                objects[obj_id]['center'] = (det[0], det[1])
                objects[obj_id]['rect'] = det[2:] # Bounding box
                objects[obj_id]['missing'] = 0 # Reinicia contador de falhas
                
                # CALCULO DA VELOCIDADE
                # Baseado na distância percorrida desde a entrada na cena (start_pos)
                start_y = objects[obj_id]['start_pos'][1]
                curr_y = det[1]
                dist_px = abs(curr_y - start_y)
                
                # Só calcula se o deslocamento for significativo (>30px) para reduzir ruído
                if dist_px > 30:
                    dist_m = dist_px / pixels_per_meter # Conversão px par metros
                    time_s = (frame_count - objects[obj_id]['start_frame']) / fps # Tempo 
                    
                    if time_s > 0:
                        speed = (dist_m / time_s) * 3.6 # m/s para km/h
                        
                        # "Filtro" para suavizar (Média Ponderada)
                        if objects[obj_id]['speed'] == 0:
                            objects[obj_id]['speed'] = speed
                        else:
                            objects[obj_id]['speed'] = (objects[obj_id]['speed'] * 0.8) + (speed * 0.2)
                
                #  LÓGICA DE CONTAGEM linha
                # Se o veículo cruzar a linha virtual (Y > 120) e ainda não foi contado
                if not objects[obj_id]['confirmed'] and det[1] > 120:
                    objects[obj_id]['confirmed'] = True
                    total_count += 1
                
                # Invalida esta linha e coluna na matriz para não serem reutilizadas
                distances[r, :] = float('inf')
                distances[:, c] = float('inf')
                used_rows.add(r)
                used_cols.add(c)

    # GESTÃO DE OBJETOS NÃO ASSOCIADOS
    
    # Objetos antigos sem correspondência (possível oclusão ou saida da cena)
    for i, obj_id in enumerate(object_ids):
        if i not in used_rows:
            objects[obj_id]['missing'] += 1

    # Deteções novas sem correspondência (novos veículos a entrar na cena)
    for j in range(len(detections)):
        if j not in used_cols:
            det = detections[j]
            objects[next_id] = {
                'center': (det[0], det[1]),
                'rect': det[2:],
                'start_pos': (det[0], det[1]),
                'start_frame': frame_count,
                'speed': 0.0,
                'missing': 0,
                'confirmed': False,
                'processed_for_avg': False
            }
            next_id += 1

    objects = cleanup_objects(objects, max_missing, img_height)
    return objects, next_id, total_count


# PRINCIPAL
# Inicialização da captura de vídeo
cap = cv2.VideoCapture('AutoEstrada.avi')
ret, frame = cap.read()
if not ret: exit()

height, width = frame.shape[:2]
fps = cap.get(cv2.CAP_PROP_FPS)

# Variaveis para estado
tracker_objects = {}
tracker_next_id = 1
tracker_total_count = 0
# Fator de calibração(ajustado para compensar a perspetiva)
PIXELS_PER_METER = 6.6 

# DEFINIÇÃO DA ZONA DE INTERESSE 
y_top, y_bot = 80, 220
x_tl, x_tr = 99, 155
x_bl, x_br = 25, 215

lanes_polys = []
lane_stats = [
    {"name": "Esq", "min_speed": 90, "total_speed": 0, "count": 0},
    {"name": "Meio", "min_speed": 70, "total_speed": 0, "count": 0},
    {"name": "Dir", "min_speed": 50, "total_speed": 0, "count": 0}
]

#  GEeo das 3 faixas de rodagem
for i in range(3):
    p1 = [get_split(x_tl, x_tr, 3, i), y_top]
    p2 = [get_split(x_tl, x_tr, 3, i+1), y_top]
    p3 = [get_split(x_bl, x_br, 3, i+1), y_bot]
    p4 = [get_split(x_bl, x_br, 3, i), y_bot]
    poly = np.array([p1, p2, p3, p4], dtype=np.int32)
    lanes_polys.append(poly)

# Criação da mascara binária 
roi_mask = np.zeros((height, width), dtype=np.uint8)
for poly in lanes_polys:
    cv2.fillPoly(roi_mask, [poly], 255)

# Pontos de referência para calibração visual para calcular px/m
calib_pt1 = (98, 170)
calib_pt2 = (90, 200)

# FUNDO
# Utiliza-se a mediana temporal para estimar o fundo estático (sem veículos)
frames_buffer = []
for _ in range(50): #50 frames
    cap.grab()
    _, f = cap.retrieve()
    if f is not None: frames_buffer.append(f)

median_frame = np.median(np.array(frames_buffer), axis=0).astype(dtype=np.uint8)
gray_background = cv2.cvtColor(median_frame, cv2.COLOR_BGR2GRAY)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_count = 0

# WHILE TRUE PARA PROCESSAMENTO
while True:
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1
    
    # 1 Pré-processamento: Conversão cinzentos
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # 1 canal
    
    # 2 Deteção de Movimento: Subtração de fundo
    diff = cv2.absdiff(gray, gray_background) 
    
    # 3 Segmentação: Thresholding para obter máscara binária
    _, mask = cv2.threshold(diff, 35, 255, cv2.THRESH_BINARY)
    
    # 4 Operações Morfológicas
    kernel = np.ones((3,3), np.uint8) 
    mask = cv2.dilate(mask, kernel, iterations=1) # Dilatação, expande as regiões brancas
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2) # Fecho tira buracos pretos
    
    # 5. Filtragem Espacial
    mask = cv2.bitwise_and(mask, mask, mask=roi_mask)
    
    # 6. Contornos
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detections = []
    for cnt in contours:
        # Filtro de Área: Ignora ruído pequeno
        if cv2.contourArea(cnt) > 400: 
            x, y, w, h = cv2.boundingRect(cnt)
            
            # Separação: Deteta e corrige oclusões
            ratio = w / float(h)
            if ratio > 2.0: 
                # Se for muito largo, divide em dois veículos
                if h > w: 
                    h_new = int(h / 2)
                    detections.append((int(x + w/2), int(y + h_new/2), x, y, w, h_new))
                    detections.append((int(x + w/2), int(y + h_new + h_new/2), x, y + h_new, w, h_new))
                else:
                    detections.append((int(x+w/2), int(y+h/2), x, y, w, h))
            else:
                detections.append((int(x+w/2), int(y+h/2), x, y, w, h))
    
    # 7. Atualização do Tracker
    tracker_objects, tracker_next_id, tracker_total_count = update_tracker(
        tracker_objects, detections, frame_count, fps, PIXELS_PER_METER, height, tracker_next_id, tracker_total_count
    )
    
    # VISUALIZAÇÃO 
    for obj_id, data in tracker_objects.items():
        if data['missing'] > 1: continue 
        
        cx, cy = data['center']
        x, y, w, h = data['rect']
        speed = int(data['speed'])
        
        # Identificação da faixa de rodagem
        lane_idx = -1
        point = (float(cx), float(cy))
        for i, poly in enumerate(lanes_polys):
            if cv2.pointPolygonTest(poly, point, False) >= 0:
                lane_idx = i
                break
        
        # Atualização das médias estatísticas por faixa
        if lane_idx != -1 and data['confirmed'] and not data['processed_for_avg'] and speed > 0:
            lane_stats[lane_idx]["total_speed"] += speed
            lane_stats[lane_idx]["count"] += 1
            data['processed_for_avg'] = True
        
        # Verificação de conformidade de velocidade
        status_text = ""
        box_color = (0, 255, 0) # Verde = Normal
        
        if lane_idx != -1 and speed > 0:
            min_s = lane_stats[lane_idx]["min_speed"]
            if speed < min_s:
                status_text = "BAIXA"
                box_color = (0, 0, 255) # Vermelho = Lento
        
        # Desenho da Bounding Box e Info
        cv2.rectangle(frame, (x, y), (x + w, y + h), box_color, 2)
        label = f"ID:{obj_id} {speed}km/h {status_text}"
        cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # Output para a consola Velocidades Médias
    console_output = f"Frame {frame_count} >> "
    for i in range(3):
        avg = 0
        if lane_stats[i]["count"] > 0:
            avg = int(lane_stats[i]["total_speed"] / lane_stats[i]["count"])
        console_output += f"{lane_stats[i]['name']}: {avg} km/h | "
    print(console_output)

   
    cv2.polylines(frame, lanes_polys, isClosed=True, color=(200, 200, 200), thickness=1)
    
    cv2.line(frame, calib_pt1, calib_pt2, (0, 255, 255), 2)
    cv2.circle(frame, calib_pt1, 3, (0, 0, 255), -1) 
    cv2.circle(frame, calib_pt2, 3, (0, 0, 255), -1)
    
    dist_px = math.hypot(calib_pt1[0] - calib_pt2[0], calib_pt1[1] - calib_pt2[1])
    cv2.putText(frame, f"Dist: {int(dist_px)}px", (calib_pt2[0] + 10, calib_pt2[1]), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Contador 
    cv2.rectangle(frame, (0, 0), (200, 40), (0, 0, 0), -1)
    cv2.putText(frame, f"Total: {tracker_total_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    cv2.imshow("TP2_51998_PIV", frame)
    #cv2.imshow("Mask", mask)
    if cv2.waitKey(30) == ord('q'): break

cap.release()
cv2.destroyAllWindows()